# BirdCLEF 2026 Training v21 — Perch Head Retrain (GCP)
## v20 scored 0.774 with perch_head  |  root cause: training/inference domain mismatch
## New in v21:
##   perch_head ONLY (resnet18 + b0 from v20 are fine — skip retrain)
##   EMBD_DIR now includes soundscape window embeddings (from precompute-gcp-v2)
##   EmbeddingDataset supports real soundscape labels (train_soundscapes_labels.csv)
##   No pseudo-labeling step — replaced by real labeled soundscape embeddings
## Same: 5-fold, BCE, 25ep, SWA, all hyperparams from v20

### Inputs required:
- birdclef-2026 (competition data)
- birdclef-2026-perch-embs-v2 (from birdclef2026-perch-precompute-gcp-v2.ipynb)

### Environment vars (GCP):
```bash
export BIRDCLEF_DATA=~/data/birdclef-2026
export BIRDCLEF_OUT=~/birdclef-output        # checkpoints saved here
export PERCH_EMBS=~/birdclef-output/perch_embeddings   # from precompute-gcp-v2
```

In [ ]:
# === CELL 1: INSTALL & IMPORTS (v21 — PyTorch only, no TF) ===
import os, json, ast, copy, random
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import autocast, GradScaler

import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

# ── CONFIG ───────────────────────────────────────────────────────────────────────────────
CFG = dict(
    sr            = 16000,
    seconds       = 5,
    n_mels        = 64,
    n_fft         = 1024,
    hop           = 320,
    fmin          = 60,
    fmax          = 8000,
    epochs        = 25,
    warmup_epochs = 4,
    lr            = 1e-3,
    batch_size    = 32,
    patience      = 7,
    num_workers   = 0,
    seed          = 42,
    mixup_alpha   = 0.1,
    spec_freq_mask = 8,
    spec_time_mask = 32,
    aug_gain_db    = 6.0,
    aug_noise_snr  = 25.0,
    aug_pitch_semi = 2,
    aug_prob       = 0.5,
    swa_start_frac = 0.6,
    soundscape_sample_weight = 0.8,  # v21: real labels, slightly higher than pseudo (0.5)
    secondary_label_weight   = 0.3,
    architectures  = ["perch_head"],  # v21: only retrain perch_head; resnet18+b0 from v20 are fine
    perch_emb_dim  = 1536,
    perch_emb_noise= 0.02,
    folds         = 5,
    device        = "cuda" if torch.cuda.is_available() else "cpu",
)

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])

print("v21 — perch_head retrain with in-domain soundscape embeddings")
print(f"   Device       : {CFG['device']}")
print(f"   Folds        : {CFG['folds']}  ({len(CFG['architectures'])*CFG['folds']} total models)")
print(f"   Epochs       : {CFG['epochs']}")
print(f"   Architectures: {CFG['architectures']}")
print(f"   AMP enabled  : {torch.cuda.is_available()}")

In [ ]:
# === CELL 2: DATA PATHS & SPECIES (GCP VERSION) ===
import os
from pathlib import Path

def _first_existing(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

# GCP root dirs (set env vars before running, or use defaults)
_data_root = str(Path(os.environ.get('BIRDCLEF_DATA', '~/data/birdclef-2026')).expanduser())
_out_root  = str(Path(os.environ.get('BIRDCLEF_OUT',  '~/birdclef-output')).expanduser())
# v21: point to v2 embeddings which contain both focal + soundscape windows
_embs_root = str(Path(os.environ.get('PERCH_EMBS', '~/birdclef-output/perch_embeddings')).expanduser())

os.makedirs(_out_root, exist_ok=True)

TRAIN_META_CSV  = _first_existing(
    f"{_data_root}/train.csv",
    "/kaggle/input/birdclef-2026/train.csv",
    "/kaggle/input/competitions/birdclef-2026/train.csv",
)
TAXONOMY_CSV    = _first_existing(
    f"{_data_root}/taxonomy.csv",
    "/kaggle/input/birdclef-2026/taxonomy.csv",
    "/kaggle/input/competitions/birdclef-2026/taxonomy.csv",
)
SOUNDSCAPE_ANNO = _first_existing(
    f"{_data_root}/train_soundscapes_labels.csv",
    "/kaggle/input/birdclef-2026/train_soundscapes_labels.csv",
    "/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv",
)

# v21: EMBD_DIR must be the v2 dataset (focal + soundscape window embeddings)
EMBD_DIR = _first_existing(
    _embs_root,                                                        # GCP: ~/birdclef-output/perch_embeddings
    "/kaggle/input/birdclef-2026-perch-embs-v2/perch_embeddings",     # Kaggle: v2 dataset
    "/kaggle/input/birdclef-2026-perch-embs-v2",
    "/kaggle/input/birdclef-2026-perch-embs/perch_embeddings",        # fallback: v1 (focal only)
    "/kaggle/input/birdclef-2026-perch-embs",
)
_all_emb_files = list(Path(EMBD_DIR).glob("*.npy")) if os.path.isdir(EMBD_DIR) else []
_focal_embs    = {f.stem for f in _all_emb_files if not f.stem.startswith("soundscape_")}
_sc_embs       = {f.stem for f in _all_emb_files if f.stem.startswith("soundscape_")}
_perch_ready   = len(_focal_embs) > 100
print(f"EMBD_DIR             : {EMBD_DIR}")
print(f"  Focal embeddings   : {len(_focal_embs)}")
_sc_label = '✅ v2 dataset' if _sc_embs else '⚠️  MISSING — run precompute-gcp-v2'
print(f"  Soundscape embeds  : {len(_sc_embs)}  ({_sc_label})")
print(f"  perch_ready        : {_perch_ready}")

if not _sc_embs:
    print()
    print("WARNING: No soundscape embeddings found. Run birdclef2026-perch-precompute-gcp-v2.ipynb first.")
    print("Training will proceed on focal clips only (same as v20 \u2014 no improvement expected).")

taxonomy_df  = pd.read_csv(TAXONOMY_CSV)
species      = taxonomy_df["primary_label"].astype(str).tolist()
species_set  = set(species)
sp_idx       = {lab: i for i, lab in enumerate(species)}
n_classes    = len(species)

df = pd.read_csv(TRAIN_META_CSV)

_species_out = f"{_out_root}/species.json"
with open(_species_out, "w") as f:
    json.dump(species, f)
print(f"species.json -> {_species_out}")

print(f"Loaded {n_classes} species | Training samples: {len(df)}")

In [ ]:
# === CELL 3: AUDIO HELPER FUNCTIONS (identical to v8) ===
def parse_secondary(s):
    if pd.isna(s): return []
    t = str(s).strip()
    if t in ("", "[]"): return []
    try:
        lst = ast.literal_eval(t)
        return [str(v) for v in lst] if isinstance(lst, list) else []
    except Exception:
        return []

def row_to_soft_multihot(primary_id: str, secondary_str: str) -> np.ndarray:
    y = np.zeros(n_classes, dtype="float32")
    p = str(primary_id)
    if p in sp_idx:
        y[sp_idx[p]] = 1.0
    for sid in parse_secondary(secondary_str):
        if sid in species_set:
            y[sp_idx[sid]] = CFG["secondary_label_weight"]
    return y

def soundscape_to_multihot(label_str: str) -> np.ndarray:
    """v21: real soundscape labels — all semicolon-separated species = 1.0."""
    y = np.zeros(n_classes, dtype="float32")
    for sp in str(label_str).split(";"):
        sp = sp.strip()
        if sp in sp_idx:
            y[sp_idx[sp]] = 1.0
    return y

def fixed_length_mono(y: np.ndarray, sr: int, seconds: int = 5) -> np.ndarray:
    target = sr * seconds
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        y = y[:target]
    return y.astype(np.float32)

def logmel_from_wave(wave: np.ndarray, sr: int) -> np.ndarray:
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr,
        n_fft=CFG["n_fft"],
        hop_length=CFG["hop"],
        n_mels=CFG["n_mels"],
        fmin=CFG["fmin"],
        fmax=CFG["fmax"],
        power=2.0,
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min, S_max = S_db.min(), S_db.max()
    if S_max - S_min < 1e-9:
        return np.zeros_like(S_db, dtype=np.float32)
    return np.clip((S_db - S_min) / (S_max - S_min + 1e-9), 0.0, 1.0).astype(np.float32)

print("\u2705 Helper functions defined (+ soundscape_to_multihot for v21 real labels)")

In [ ]:
# === CELL 4: MEL PRECOMPUTE — skipped (v21 trains perch_head only, no mel models) ===
print("v21: skipping mel precompute \u2014 perch_head only, no mel-based models to train")
print("  resnet18 + efficientnet_b0 checkpoints come from v20 (birdclef-2026-v20-model)")

In [ ]:
# === CELL 5: LOSS + SPECAUGMENT + AUDIO AUGMENTATIONS (v18) ===

criterion = nn.BCEWithLogitsLoss()
print(f"Loss: BCEWithLogitsLoss (no focal)")

def spec_augment(mel: torch.Tensor, freq_mask: int, time_mask: int) -> torch.Tensor:
    _, F, T = mel.shape
    mel = mel.clone()
    if freq_mask > 0 and F > freq_mask:
        f = random.randint(0, freq_mask)
        f0 = random.randint(0, F - f) if f > 0 else 0
        mel[:, f0:f0 + f, :] = 0.0
    if time_mask > 0 and T > time_mask:
        t = random.randint(0, time_mask)
        t0 = random.randint(0, T - t) if t > 0 else 0
        mel[:, :, t0:t0 + t] = 0.0
    return mel

print(f"SpecAugment: freq_mask={CFG['spec_freq_mask']} bins, time_mask={CFG['spec_time_mask']} frames")

In [ ]:
# === CELL 6: MODEL ARCHITECTURES ===
class BirdCLEFModel(nn.Module):
    """Kept for reference / loading v20 mel-based checkpoints (not trained in v21)."""
    def __init__(self, arch: str, n_classes: int, pretrained: bool = True):
        super().__init__()
        self.arch = arch
        if arch == "resnet18":
            self.backbone = timm.create_model("resnet18", pretrained=pretrained, num_classes=0)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            in_features = self.backbone.num_features
        elif arch in ("efficientnet_b0", "efficientnet_b2"):
            self.backbone = timm.create_model(arch, pretrained=pretrained, num_classes=0)
            stem = self.backbone.conv_stem
            self.backbone.conv_stem = nn.Conv2d(
                1, stem.out_channels,
                kernel_size=stem.kernel_size, stride=stem.stride,
                padding=stem.padding, bias=False,
            )
            in_features = self.backbone.num_features
        else:
            raise ValueError(f"Unsupported arch: {arch}")
        self.head = nn.Sequential(
            nn.Linear(in_features, 512), nn.ReLU(), nn.Dropout(0.4), nn.Linear(512, n_classes),
        )
    def forward(self, x): return self.head(self.backbone(x))

device = torch.device(CFG["device"])
print("\u2705 BirdCLEFModel defined (not trained in v21 \u2014 kept for compat)\n")

# \u2500\u2500 PerchHead (v19+): MLP on top of frozen Perch 1536-dim embeddings \u2500\u2500──────────────────
class PerchHead(nn.Module):
    """Lightweight classifier trained on precomputed Perch 2.0 embeddings."""
    def __init__(self, n_classes: int, emb_dim: int = 1536):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, n_classes),
        )
    def forward(self, x): return self.net(x)  # x: (B, 1536)

_ph = PerchHead(n_classes=n_classes)
print(f"PerchHead: {sum(p.numel() for p in _ph.parameters())/1e6:.2f}M parameters")
del _ph

In [ ]:
# === CELL 7: PREPARE SOUNDSCAPE EMBEDDING ROWS (v21 — replaces pseudo-labeling) ===
# v20 used pseudo-labels generated by running v15 models on raw soundscape audio.
# v21 uses REAL labels from train_soundscapes_labels.csv, paired with pre-computed
# Perch embeddings extracted at the exact labeled timestamps (soundscape_*_*s.npy).

def _parse_hms_to_secs(s: str) -> int:
    parts = str(s).strip().split(":")
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])

sc_anno = pd.read_csv(SOUNDSCAPE_ANNO)
sc_rows = []
missing_sc_embs = 0

for _, row in sc_anno.iterrows():
    stem     = Path(str(row["filename"])).stem
    end_secs = _parse_hms_to_secs(row["end"])
    emb_stem = f"soundscape_{stem}_{end_secs}s"

    if emb_stem not in _sc_embs:
        missing_sc_embs += 1
        continue

    # Derive primary_label for stratification (first valid species in the list)
    labels_list = [s.strip() for s in str(row["primary_label"]).split(";") if s.strip() in species_set]
    if not labels_list:
        continue

    sc_rows.append({
        "filename"     : emb_stem,
        "primary_label": labels_list[0],
        "all_labels"   : row["primary_label"],   # full semicolon-separated list
        "label_mode"   : "soundscape",
        "secondary_labels": "[]",
        "sample_weight": CFG["soundscape_sample_weight"],
    })

sc_emb_df = pd.DataFrame(sc_rows) if sc_rows else pd.DataFrame(
    columns=["filename", "primary_label", "all_labels", "label_mode", "secondary_labels", "sample_weight"]
)
print(f"Soundscape embedding rows : {len(sc_emb_df)}")
print(f"  (missing emb files      : {missing_sc_embs} \u2014 run precompute-gcp-v2 to fix)")
print(f"  Species covered         : {sc_emb_df['primary_label'].nunique() if len(sc_emb_df) else 0}")

In [ ]:
# === CELL 8: DATASET WITH MIXUP ===
class ClipDataset(Dataset):
    """Mel-spectrogram dataset — unchanged from v8 (kept for compat, not used in v21)."""
    def __init__(self, frame, mel_root, train):
        self.df = frame.reset_index(drop=True)
        self.mel_root = Path(mel_root)
        self.train = train
        self.win_frames = int(CFG["seconds"] * CFG["sr"] / CFG["hop"]) + 1
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        mel = np.load(self.mel_root / (str(r["filename"]).replace("/", "_") + ".npy")).astype("float32")
        T, W = mel.shape[1], self.win_frames
        if T <= W:
            mel = np.concatenate([mel, np.zeros((mel.shape[0], W - T), dtype=np.float32)], axis=1)
        else:
            start = np.random.randint(0, T - W) if self.train else (T - W) // 2
            mel = mel[:, start:start + W]
        x = torch.from_numpy(mel[None]).float()
        y = torch.from_numpy(row_to_soft_multihot(r["primary_label"], r.get("secondary_labels", "[]"))).float()
        w = torch.tensor(float(r["sample_weight"]) if "sample_weight" in self.df.columns else 1.0)
        return x, y, w

def mixup_collate(batch, alpha: float = 0.1, use_mixup: bool = True):
    xs, ys, ws = zip(*batch)
    xs, ys, ws = torch.stack(xs), torch.stack(ys), torch.stack(ws)
    if not use_mixup or alpha <= 0:
        return xs, ys, ws
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(xs.size(0))
    return lam * xs + (1 - lam) * xs[idx], lam * ys + (1 - lam) * ys[idx], lam * ws + (1 - lam) * ws[idx]

print("\u2705 ClipDataset defined (not used in v21)")

# \u2500\u2500 EmbeddingDataset (v21): focal + soundscape rows \u2500\u2500────────────────────────────
class EmbeddingDataset(Dataset):
    """
    Loads precomputed Perch embeddings.
    Handles two row types:
      label_mode == 'focal'      -> row_to_soft_multihot (primary + secondary)
      label_mode == 'soundscape' -> soundscape_to_multihot(all_labels)  [v21]
    """
    def __init__(self, frame: pd.DataFrame, emb_root: str, train: bool):
        self.df       = frame.reset_index(drop=True)
        self.emb_root = Path(emb_root)
        self.train    = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r        = self.df.iloc[i]
        fname    = str(r["filename"])
        emb_name = fname if fname.endswith(".npy") else fname + ".npy"
        emb      = np.load(self.emb_root / emb_name).astype("float32")
        x        = torch.from_numpy(emb)  # (1536,)
        # Light augmentation on embedding during training
        if self.train and random.random() < CFG["aug_prob"]:
            x = x + torch.randn_like(x) * CFG["perch_emb_noise"]
        # v21: choose label builder by row type
        if str(r.get("label_mode", "focal")) == "soundscape":
            y_np = soundscape_to_multihot(r["all_labels"])
        else:
            y_np = row_to_soft_multihot(r["primary_label"], r.get("secondary_labels", "[]"))
        y = torch.from_numpy(y_np).float()
        w = torch.tensor(float(r["sample_weight"]) if "sample_weight" in self.df.columns else 1.0)
        return x, y, w


def make_emb_loader(frame, train: bool):
    ds = EmbeddingDataset(frame, EMBD_DIR, train=train)
    return DataLoader(
        ds,
        batch_size=CFG["batch_size"] * 4,  # embeddings are small: use larger batch
        shuffle=train,
        num_workers=CFG["num_workers"],
        collate_fn=lambda b: mixup_collate(b, CFG["mixup_alpha"], use_mixup=train),
        drop_last=train,
    )

print("\u2705 EmbeddingDataset (v21: focal + soundscape) and make_emb_loader defined")

In [ ]:
# === CELL 9: PREPARE TRAINING DATA — focal embeddings + real soundscape embeddings ===

# --- Focal clips ---
emb_train_df = df.copy()
emb_train_df["filename"]      = emb_train_df["filename"].apply(lambda x: x.replace("/", "_"))
emb_train_df["label_mode"]    = "focal"
emb_train_df["all_labels"]    = emb_train_df["primary_label"]
emb_train_df["sample_weight"] = 1.0
if "secondary_labels" not in emb_train_df.columns:
    emb_train_df["secondary_labels"] = "[]"
else:
    emb_train_df["secondary_labels"] = emb_train_df["secondary_labels"].fillna("[]")
emb_train_df = emb_train_df[emb_train_df["filename"].isin(_focal_embs)].reset_index(drop=True)
print(f"Focal embedding samples : {len(emb_train_df)}")

# --- Combine with real soundscape rows ---
_cols = ["filename", "primary_label", "all_labels", "label_mode", "secondary_labels", "sample_weight"]
combined_emb_df = pd.concat(
    [emb_train_df[_cols], sc_emb_df[_cols]],
    ignore_index=True,
)
print(f"Soundscape rows added   : {len(sc_emb_df)}")
print(f"Combined (emb) total    : {len(combined_emb_df)}")

print(f"\n\u2705 Training setup:")
print(f"   Total emb samples  : {len(combined_emb_df)}  (focal={len(emb_train_df)}, soundscape={len(sc_emb_df)})")
print(f"   Classes            : {n_classes}")
print(f"   Device             : {device}")

In [ ]:
# === CELL 10: 5-FOLD PERCH_HEAD TRAINING + AMP + SWA — v21 ===
n_models = len(CFG["architectures"]) * CFG["folds"]
print("=" * 70)
print(f"TRAINING: {n_models} models  ({CFG['folds']} folds x {len(CFG['architectures'])} architectures)  v21")
print("=" * 70)

_use_amp = (device.type == "cuda")
print(f"   Mixed precision (AMP) : {'ENABLED' if _use_amp else 'disabled (CPU mode)'}")

_criterion      = nn.BCEWithLogitsLoss(reduction='none')
_criterion_mean = nn.BCEWithLogitsLoss()
skf             = StratifiedKFold(n_splits=CFG["folds"], shuffle=True, random_state=CFG["seed"])

oof_preds   = {arch: np.zeros((len(combined_emb_df), n_classes), dtype=np.float32)
               for arch in CFG["architectures"]}
arch_scores = {arch: [] for arch in CFG["architectures"]}

for arch in CFG["architectures"]:
    print(f"\n{'='*60}")
    print(f"ARCHITECTURE: {arch}")
    print(f"{'='*60}")

    if arch == "perch_head" and not _perch_ready:
        print(f"  Skipping {arch} \u2014 Perch embeddings not available")
        continue

    # v21: always use combined embedding dataframe for perch_head
    _cur_df         = combined_emb_df
    _make_loader_fn = make_emb_loader

    for fold_idx, (train_idx, val_idx) in enumerate(
        skf.split(_cur_df, _cur_df["primary_label"].apply(
            lambda x: x if _cur_df["primary_label"].value_counts().get(x, 0) >= CFG["folds"] else "__rare__"
        ))
    ):
        print(f"\n  Fold {fold_idx + 1}/{CFG['folds']}")

        train_loader = _make_loader_fn(_cur_df.iloc[train_idx], train=True)
        val_loader   = _make_loader_fn(_cur_df.iloc[val_idx],   train=False)

        model     = PerchHead(n_classes=n_classes, emb_dim=CFG["perch_emb_dim"]).to(device)
        optimizer = AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-4)
        scaler    = GradScaler(enabled=_use_amp)

        warmup_sched = LinearLR(optimizer, start_factor=0.1, end_factor=1.0,
                                total_iters=CFG["warmup_epochs"])
        cosine_sched = CosineAnnealingLR(optimizer,
                                         T_max=max(1, CFG["epochs"] - CFG["warmup_epochs"]),
                                         eta_min=1e-6)
        scheduler    = SequentialLR(optimizer,
                                    schedulers=[warmup_sched, cosine_sched],
                                    milestones=[CFG["warmup_epochs"]])

        best_val_auc    = -1.0
        patience_count  = 0
        best_state      = None
        best_fold_preds = None
        swa_states      = []
        _swa_start_ep   = max(1, int(CFG["epochs"] * CFG["swa_start_frac"]))

        for epoch in range(CFG["epochs"]):
            # Train
            model.train()
            train_loss = 0.0
            for x, y, w in train_loader:
                x, y, w = x.to(device), y.to(device), w.to(device)
                optimizer.zero_grad()
                with autocast(enabled=_use_amp):
                    logits_tr = model(x)
                    loss = (_criterion(logits_tr, y).mean(dim=1) * w).mean()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                train_loss += loss.item()
            train_loss /= max(len(train_loader), 1)
            scheduler.step()

            # Validate
            model.eval()
            val_loss   = 0.0
            fold_preds, fold_targets = [], []
            with torch.no_grad():
                for x, y, _w in val_loader:
                    x, y = x.to(device), y.to(device)
                    with autocast(enabled=_use_amp):
                        logits = model(x)
                    logits_f = logits.float()
                    val_loss += _criterion_mean(logits_f, y).item()
                    fold_preds.append(torch.sigmoid(logits_f).cpu().numpy())
                    fold_targets.append(y.cpu().numpy())
            val_loss /= max(len(val_loader), 1)

            if not fold_preds:
                val_auc = 0.0
            else:
                fp     = np.vstack(fold_preds)
                ft     = np.vstack(fold_targets)
                ft_bin = (ft >= 0.5).astype(np.float32)
                auc_scores_ep = [
                    roc_auc_score(ft_bin[:, j], fp[:, j])
                    for j in range(n_classes)
                    if ft_bin[:, j].sum() > 0 and (1 - ft_bin[:, j]).sum() > 0
                ]
                val_auc = np.mean(auc_scores_ep) if auc_scores_ep else 0.0

            cur_lr = scheduler.get_last_lr()[0]

            if val_auc > best_val_auc:
                best_val_auc    = val_auc
                patience_count  = 0
                best_state      = copy.deepcopy(model.state_dict())
                best_fold_preds = fp.copy() if fold_preds else np.zeros((len(val_idx), n_classes), dtype=np.float32)
            else:
                patience_count += 1

            if epoch >= _swa_start_ep:
                swa_states.append(copy.deepcopy(model.state_dict()))
                if len(swa_states) > 10:
                    swa_states.pop(0)

            if (epoch + 1) % 5 == 0 or patience_count == 0:
                print(f"    Epoch {epoch+1:3d}: train={train_loss:.4f}  "
                      f"val={val_loss:.4f}  auc={val_auc:.4f}  lr={cur_lr:.2e}")

            if patience_count >= CFG["patience"]:
                print(f"    Early stopping at epoch {epoch+1}")
                break

        if best_state is None:
            print(f"  Warning: no best_state for fold {fold_idx} \u2014 saving current weights")
            best_state      = copy.deepcopy(model.state_dict())
            best_fold_preds = np.zeros((len(val_idx), n_classes), dtype=np.float32)

        # SWA: average swa_states if available
        if swa_states:
            avg_state = {}
            for k in swa_states[0].keys():
                avg_state[k] = torch.stack(
                    [s[k].float() for s in swa_states]
                ).mean(dim=0).to(swa_states[0][k].dtype)
            model.load_state_dict(avg_state)
            print(f"  SWA: averaged {len(swa_states)} checkpoints")
        else:
            model.load_state_dict(best_state)

        # v21: save as perch_head_v21_fold{i}.pt
        ckpt_path = os.path.join(_out_root, f"{arch}_v21_fold{fold_idx}.pt")
        torch.save(model.state_dict(), ckpt_path)

        oof_preds[arch][val_idx] = best_fold_preds
        arch_scores[arch].append(best_val_auc)
        print(f"  Fold {fold_idx+1} best AUC: {best_val_auc:.4f}  saved {ckpt_path}")
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_auc = np.mean(arch_scores[arch])
    print(f"\n  {arch}: mean OOF AUC = {mean_auc:.4f}")

print(f"\nAll {n_models} perch_head models trained!")

In [ ]:
# === CELL 11: SUMMARY & UPLOAD ===
print("=" * 70)
print("TRAINING SUMMARY v21")
print("=" * 70)

for arch in CFG["architectures"]:
    fold_aucs = arch_scores[arch]
    if fold_aucs:
        print(f"\n{arch}:")
        print(f"   Fold AUCs : {[f'{a:.4f}' for a in fold_aucs]}")
        print(f"   Mean AUC  : {np.mean(fold_aucs):.4f} \u00b1 {np.std(fold_aucs):.4f}")

print(f"\n\u2705 Saved checkpoints:")
for arch in CFG["architectures"]:
    for fold_idx in range(CFG["folds"]):
        print(f"   {_out_root}/{arch}_v21_fold{fold_idx}.pt")

print()
print("Next: upload to Kaggle as birdclef-2026-v21-model")
print("  kaggle datasets create -p <upload_dir> --dir-mode zip")
print()
print("Then in inference notebook:")
print("  1. Add birdclef-2026-v21-model as input")
print("  2. Set CKPT_DIR_V21 = '/kaggle/input/birdclef-2026-v21-model'")
print("  3. Re-enable 'perch_head' in architectures (load v21 checkpoints for it)")